# t-SNE — Visualizing Data using t-SNE (2008)

_Papers · Classical ML_

**Turn high-dimensional data into a 2-D picture where true neighbors stay together and clusters pull apart.**

---

This notebook is a practice scaffold for the **t-SNE — Visualizing Data using t-SNE (2008)** lesson. We build t-SNE one piece at a time: the high-D affinities, the low-D Student-t affinities and gradient, then the full optimization loop checked against scikit-learn. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — NumPy

### Step 1 — Compute the high-dimensional affinities

t-SNE first turns the raw high-D distances into conditional probabilities `p_{j|i}`: a Gaussian centered on each point `i`, whose width `sigma_i` (here `beta = 1/(2 sigma^2)`) is tuned so the distribution has a target **perplexity** — an effective neighbor count (Eq. 1). `_Hbeta` returns the row probabilities and their entropy for a given `beta`, and `high_dim_affinities` runs a binary search on `beta` per row until the entropy matches `log(perplexity)`. Finally the conditionals are symmetrized into the joint distribution `p_{ij}` (Section 3.1).

In [ ]:
import numpy as np
np.random.seed(0)

# ---------- high-D affinities: Gaussian p_{j|i} with a perplexity binary search (Eq. 1) ----------
def _Hbeta(Drow, beta):
    """Row probabilities P and entropy H for a given beta = 1/(2 sigma^2)."""
    P = np.exp(-Drow * beta)
    sumP = P.sum()
    H = np.log(sumP) + beta * np.sum(Drow * P) / sumP   # entropy in nats
    return H, P / sumP

def high_dim_affinities(X, target_perplexity=30.0, tol=1e-5, max_iter=50):
    n = X.shape[0]
    sumX = np.sum(X * X, axis=1)
    D = sumX[:, None] + sumX[None, :] - 2 * X @ X.T     # squared distances ||x_i - x_j||^2
    P = np.zeros((n, n))
    logU = np.log(target_perplexity)                    # perplexity = 2^H  ->  H = log2(perp)  (nats: log perp)
    for i in range(n):
        beta = 1.0
        lo, hi = -np.inf, np.inf
        idx = np.concatenate((np.arange(i), np.arange(i + 1, n)))   # k != i
        Drow = D[i, idx]
        for _ in range(max_iter):                       # binary search beta until perplexity matches
            H, thisP = _Hbeta(Drow, beta)
            if abs(H - logU) < tol:
                break
            if H > logU:                                # too spread out -> shrink sigma (raise beta)
                lo = beta
                beta = beta * 2 if hi == np.inf else (beta + hi) / 2
            else:                                       # too peaked -> grow sigma (lower beta)
                hi = beta
                beta = beta / 2 if lo == -np.inf else (beta + lo) / 2
        P[i, idx] = thisP                               # this is p_{j|i}
    P = (P + P.T) / (2 * n)                              # symmetrize -> joint p_{ij}  (Section 3.1)
    return np.maximum(P, 1e-12)

### Step 2 — Define the low-D affinities and the optimization loop

In the 2-D map, affinities `q_{ij}` use a heavy-tailed **Student-t** kernel `(1 + d^2)^{-1}` (Eq. 4), which lets moderately distant points keep a non-zero affinity and avoids crowding. The `tsne` loop minimizes the KL divergence between `P` and `Q` by gradient descent (Eq. 5), with momentum, an **early exaggeration** phase (multiply `P` by 4 for the first 100 steps to form tight clusters), and re-centering each step. The gradient is the familiar attractive/repulsive 'spring' force: `4 * sum_j (p_ij - q_ij)(y_i - y_j)(1+||y_i-y_j||^2)^{-1}`.

In [ ]:
# ---------- low-D Student-t q_{ij} (Eq. 4) and KL gradient (Eq. 5) ----------
def low_dim_q(Y):
    sumY = np.sum(Y * Y, axis=1)
    dist = sumY[:, None] + sumY[None, :] - 2 * Y @ Y.T  # ||y_i - y_j||^2
    num = 1.0 / (1.0 + dist)                            # (1 + d^2)^{-1}
    np.fill_diagonal(num, 0.0)                          # q_{ii} = 0
    Q = np.maximum(num / num.sum(), 1e-12)
    return Q, num

def tsne(X, no_dims=2, target_perplexity=30.0, n_iter=600, lr=200.0, seed=0):
    rng = np.random.RandomState(seed)
    n = X.shape[0]
    P = high_dim_affinities(X, target_perplexity)
    P *= 4.0                                            # early exaggeration
    Y = rng.randn(n, no_dims) * 1e-4
    vel = np.zeros_like(Y)
    for it in range(n_iter):
        Q, num = low_dim_q(Y)
        PQ = P - Q
        # Eq. 5: dC/dy_i = 4 sum_j (p_ij - q_ij)(y_i - y_j)(1+||y_i-y_j||^2)^{-1}
        diff = Y[:, None, :] - Y[None, :, :]            # pairwise (y_i - y_j) vectors
        weighted = (PQ * num)[:, :, None] * diff
        grad = 4.0 * weighted.sum(axis=1)
        momentum = 0.5 if it < 250 else 0.8
        vel = momentum * vel - lr * grad
        Y = Y + vel
        Y = Y - Y.mean(axis=0)                          # keep map centered
        if it == 100:
            P /= 4.0                                    # stop early exaggeration
    return Y

### Step 3 — Check against scikit-learn and recompute the worked example

To sanity-check the from-scratch implementation, we run it on a 3-class digits subset and measure how cleanly the classes separate with a leave-one-out 3-NN accuracy in the 2-D map — then compare to `sklearn.manifold.TSNE`. Both should land near 0.97–0.99, confirming the same qualitative clustering even though the map never sees the labels. We also recompute the small worked example (three fixed map points) so the practice problems have concrete `q` and gradient values to match.

In [ ]:
# ---------- ORACLE (qualitative): same clusters as sklearn.manifold.TSNE on a digits subset ----------
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier

digits = load_digits()
mask = np.isin(digits.target, [0, 1, 2])               # 3 classes -> readable 3-blob map
X = digits.data[mask][:150]
y = digits.target[mask][:150]
X = (X - X.mean(0)) / (X.std(0) + 1e-8)                # standardize 64-D pixels

def cluster_score(emb, labels):
    """Leave-one-out 3-NN label accuracy in the 2-D map = 'how cleanly do classes separate?'"""
    acc = 0
    knn = KNeighborsClassifier(n_neighbors=3)
    for i in range(len(emb)):
        tr = np.arange(len(emb)) != i
        knn.fit(emb[tr], labels[tr])
        acc += int(knn.predict(emb[i:i+1])[0] == labels[i])
    return acc / len(emb)

Y_mine = tsne(X, target_perplexity=30.0, seed=0)
Y_skl  = TSNE(n_components=2, perplexity=30.0, init="random", random_state=0).fit_transform(X)
print("ours  3-NN cluster accuracy :", round(cluster_score(Y_mine, y), 3))   # ~0.97
print("sklearn 3-NN cluster acc.   :", round(cluster_score(Y_skl,  y), 3))   # ~0.99
print("both separate the 3 digit classes -> same qualitative clustering")

# ---------- recompute the WORKED EXAMPLE: y1=(0,0), y2=(1,0), y3=(0,2) ----------
Yex = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 2.0]])
Qex, numex = low_dim_q(Yex)
print("worked q_12 :", round(Qex[0, 1], 5))            # 0.28846
print("worked q_13 :", round(Qex[0, 2], 5))            # 0.11538
# one gradient summand for y1 from pair (1,2), with p_12 = 0.4:
p12 = 0.4
summand = 4 * (p12 - Qex[0, 1]) * numex[0, 1] * (Yex[0] - Yex[1])
print("worked grad term y1<-y2 :", [round(v, 5) for v in summand])   # [-0.22308, 0.0]

## Visualize the data & results

_Run our from-scratch t-SNE on a 3-class handwritten-digits subset (64-D pixels) — do the three digit classes separate into three blobs in 2-D, the same way sklearn's t-SNE does, without ever being told the labels?_

### Step 1 — Run our t-SNE on a small digits subset

We take 60 examples from three digit classes, standardize the 64-D pixels, and run our from-scratch `tsne` to get a 2-D embedding `Y`. This reuses the `high_dim_affinities`, `low_dim_q`, and `tsne` functions defined in the reference cell above. Because the map is built purely from pairwise affinities, it never sees the labels `y`.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier
# (high_dim_affinities, low_dim_q, tsne defined as in the CODE cell above)

digits = load_digits()
mask = np.isin(digits.target, [0, 1, 2])
X = digits.data[mask][:60]
y = digits.target[mask][:60]
X = (X - X.mean(0)) / (X.std(0) + 1e-8)

Y = tsne(X, target_perplexity=30.0, n_iter=600, seed=0)   # our from-scratch t-SNE

### Step 2 — Score the separation and read off the cluster centers

To quantify how well the three classes separated, we compute a leave-one-out 3-NN label accuracy directly in the 2-D map — high accuracy means same-digit points landed near each other. Then we print each digit class's center coordinates and point count, summarizing the three blobs you would see plotted. An accuracy near 0.97 confirms the classes pulled apart cleanly without ever being told the labels.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)                 # cluster quality = 3-NN label accuracy
acc = 0
for i in range(len(Y)):
    tr = np.arange(len(Y)) != i
    knn.fit(Y[tr], y[tr])
    acc += int(knn.predict(Y[i:i+1])[0] == y[i])
print("3-NN cluster accuracy:", round(acc / len(Y), 3))   # ~0.97 -> classes separate

for cls in [0, 1, 2]:                                      # the labeled 2-D coordinates plotted above
    pts = Y[y == cls]
    center_x = pts[:, 0].mean()
    center_y = pts[:, 1].mean()
    print(f"digit {cls}: center=({center_x:.2f}, {center_y:.2f}), n={len(pts)}")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** For map points $y_1=(0,0)$, $y_2=(1,0)$, $y_3=(0,2)$, compute the Student-t $q_{13}$ (Eq. 4).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Squared distances: $d_{12}^2=1$, $d_{13}^2=4$, $d_{23}^2=5$. — _Eq. 4 uses $\|y_i-y_j\|^2$._
- Unnormalized weights $(1+d^2)^{-1}$: $0.5$, $0.2$, $0.16667$; normalizer $=2(0.5+0.2+0.16667)=1.73333$. — _Each unordered pair is counted twice in $\sum_{k\neq l}$._
- $q_{13}=0.2/1.73333$. — _Divide this pair's weight by the total._

**Answer:** $q_{13}\approx 0.11538$. The far pair (1,3) gets less probability than the close pair (1,2) at $0.288$, but because the tail is heavy it is still a real, non-zero affinity — not crushed to near-zero as a Gaussian would.

</details>

**Problem 2.** Using Eq. 5, compute the gradient contribution to $y_1$ from its pair with $y_3$, given $p_{13}=0.1$ and the $q_{13}\approx0.11538$ from the previous question. (Use $y_1=(0,0)$, $y_3=(0,2)$, $d_{13}^2=4$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scalar factor $4(p_{13}-q_{13})(1+d_{13}^2)^{-1}=4(0.1-0.11538)(0.2)$. — _The Eq. 5 summand without the direction vector._
- $=4(-0.01538)(0.2)=-0.01231$. — _$p_{13}\lt q_{13}$, so the scalar is negative._
- Times $(y_1-y_3)=(0,-2)$: $(-0.01231)\cdot(0,-2)=(0,\,0.02462)$. — _The summand is scalar $\times (y_i-y_j)$._

**Answer:** Contribution $\approx(0,\,0.02462)$. Because $p_{13}\lt q_{13}$ the map over-represents this pair, so the spring repels: it pushes $y_1$ in the $+y$ direction, away from $y_3$ which sits above it. (Moving $-\delta C/\delta y_1$ would instead pull them — sign care matters; here we report the raw gradient summand.)

</details>

**Problem 3.** Ablation — perplexity. In the CODE/CODEVIZ, re-run our from-scratch t-SNE on the digits subset with perplexity set very low (e.g. 2) and very high (e.g. 50) instead of the default ~30. What do you expect to see, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set the target perplexity to 2 and re-run; inspect the map. — _Perplexity is the effective neighbor count each Gaussian $\sigma_i$ is tuned to._
- Set it to 50 (close to $n$) and re-run. — _Now each point's Gaussian spans almost the whole set._
- Compare the kNN label-accuracy of each map to the default. — _A single number for 'how cleanly do classes separate?'_

**Answer:** Low perplexity (2) makes each point see only a couple of neighbors, so the map fragments — classes break into many small sub-clumps and the global picture gets noisy. High perplexity (50) makes each Gaussian span most of the data, blurring local structure so distinct digit classes start to merge. A middle value (~30) gives the cleanest three-blob separation, matching the paper's advice of perplexity 5-50 (Section 2). The kNN accuracy is typically highest in the middle and drops at both extremes.

</details>